# Lesson 2 — Value functions and the Bellman equations

*Chapter I. RL foundations and tabular methods · Section 2 of the lecture notes*

**Foundations:** Silver Lecture 2; Sutton–Barto §§3.5–3.7.

> **Learning goal.** Compute $v_\pi, q_\pi$ and explain why Bellman links present value with the future; distinguish evaluation from optimality.

> **How to read this notebook.** Markdown cells are the lecture, translated to English without changing its content. Boxes that start with **Code** map it to `bellman.py`: the class `BellmanEquations` writes Eqs. (5)–(8) with explicit sums, and `lecture_example_mdp()` builds the example of Section 4.

In [1]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

from rlcore import evaluate_policy, TabularPolicy
from bellman import BellmanEquations, lecture_example_mdp, uniform_policy

mdp = lecture_example_mdp(v_u=4.0, gamma=0.9)
bell = BellmanEquations(mdp)
pi = uniform_policy(mdp)                      # chooses a and b evenly in s
s, u = mdp.state_index("s"), mdp.state_index("u")
print(mdp, "| states:", mdp.state_names, "| actions:", mdp.action_names)

TabularMDP(n_states=3, n_actions=2, gamma=0.9) | states: ['s', 'u', 'T'] | actions: ['a', 'b']


## 1. Quick intuition

Value does not tell whether a state is good or bad in general; it tells the expected return when following *a specific policy*. $q_\pi$ additionally fixes the first action, then continues following $\pi$. Bellman turns a long sum of rewards into a relation between neighboring states, thanks to identity (3). [Silver, Lecture 2]

## 2. The problem being solved

**Prediction:** the policy is given, we need $v_\pi/q_\pi$. **Control:** we need a policy with a large return. The model may be known or unknown; Bellman is a relation that holds for the MDP, and it does not require the agent to already know the model in order to define it. A value output is different from an action output; to act, a rule for choosing actions must be added.

## 3. Core idea and formulas

$$v_\pi(s) = \mathbb E_\pi[G_t \mid S_t = s], \qquad q_\pi(s,a) = \mathbb E_\pi[G_t \mid S_t = s, A_t = a]. \tag{4}$$

**How good is it to stand at $s$?** $v_\pi(s)$ takes the expected return if from there we act according to $\pi$. **What if we are forced to choose $a$ first?** $q_\pi(s,a)$ fixes the first action, then follows $\pi$ at the subsequent steps. The notation after the sign $\mid$ is the information held fixed; the remaining randomness is still averaged. Both depend on the policy: the same state can have different values under two policies.

> **Code.** Eq. (4) is an expectation of returns, so it can be estimated by averaging $G_0$ over rollouts that start in $s$ (here the start state is $s$). `mdp.evaluate_policy_exact` solves the Bellman system exactly for comparison.

In [2]:
rng = np.random.default_rng(0)
for name, table in [("uniform", pi), ("always a", np.tile([1.0, 0.0], (3, 1))), ("always b", np.tile([0.0, 1.0], (3, 1)))]:
    mc = evaluate_policy(mdp, TabularPolicy(table), 20_000, rng).mean()
    print(f"policy {name:9s}: average of G_0 = {mc:.3f}   exact v_pi(s) = {mdp.evaluate_policy_exact(table)[s]:.3f}")

policy uniform  : average of G_0 = 3.851   exact v_pi(s) = 3.850


policy always a : average of G_0 = 4.695   exact v_pi(s) = 4.700


policy always b : average of G_0 = 3.000   exact v_pi(s) = 3.000


$$v_\pi(s) = \mathbb E_\pi[R_{t+1} + \gamma v_\pi(S_{t+1}) \mid S_t = s]. \tag{5}$$

Read the right-hand side as: **reward of the next step + discounted expected value of the future**. After reaching $S_{t+1}$, we still continue following $\pi$, so we use $v_\pi(S_{t+1})$. The expectation averages over both the actions the policy chooses and the environment's transition/reward. So there is no need to enumerate every long trajectory to describe value: it is enough to connect the present value with the next-state value. A single transition is one sample of the right-hand side, not that expectation itself.

> **Code.** `bell.sampled_target` draws one transition and returns one sample of the right-hand side. Single samples scatter; only their average approaches $v_\pi(s)$.

In [3]:
v = mdp.evaluate_policy_exact(pi)
samples = np.array([bell.sampled_target(pi, v, s, rng) for _ in range(5000)])
print("possible one-step samples:", np.unique(samples.round(3)))
print(f"mean of 5000 samples = {samples.mean():.3f}   v_pi(s) = {v[s]:.3f}")

possible one-step samples: [2.  3.  5.6]
mean of 5000 samples = 3.834   v_pi(s) = 3.850


$$q_\pi(s,a) = \mathcal R^a_s + \gamma \sum_{s'} \mathcal P^a_{ss'} v_\pi(s'). \tag{6}$$

The first action is fixed to $a$, so the average reward is $\mathcal R^a_s$. The environment may take us to many $s'$: take $v_\pi(s')$ of each place, multiply by the probability $\mathcal P^a_{ss'}$, then add them up. Finally multiply by $\gamma$. **Why sum over $s'$?** We do not know yet which next state will occur. The terminal has value zero, so a branch going to the terminal only contributes the immediate reward.

$$v_\pi(s) = \sum_a \pi(a \mid s) q_\pi(s,a). \tag{7}$$

At state $s$, the policy may choose several actions. Each action has an expected return $q_\pi(s,a)$ and a probability $\pi(a \mid s)$ of being chosen. So $v_\pi(s)$ is the **weighted average** of the action-values. If the policy chooses one action for sure, only the value of that action remains; if it chooses two actions evenly, the value is the average of the two $q$-values. This is averaging according to the current policy, not choosing the best action.

> **Code.** `bell.q_from_v` loops over $s, a, s'$ exactly as in Eq. (6); `BellmanEquations.v_from_q` is Eq. (7). `bell.evaluate` repeats Eq. (5) = Eq. (6) then Eq. (7) until nothing changes.

In [4]:
v = bell.evaluate(pi)
q = bell.q_from_v(v)
print(f"q_pi(s,a) = {q[s, 0]:.3f}   q_pi(s,b) = {q[s, 1]:.3f}   v_pi(s) = {v[s]:.3f}")
for name, table in [("uniform", pi), ("always a", np.tile([1.0, 0.0], (3, 1))), ("always b", np.tile([0.0, 1.0], (3, 1)))]:
    print(f"policy {name:9s}: v_pi(s) = {BellmanEquations.v_from_q(table, q)[s]:.3f}")

q_pi(s,a) = 4.700   q_pi(s,b) = 3.000   v_pi(s) = 3.850
policy uniform  : v_pi(s) = 3.850
policy always a : v_pi(s) = 4.700
policy always b : v_pi(s) = 3.000


Bellman expectation keeps the policy fixed. Bellman optimality replaces averaging the actions by the policy with choosing the best action:

$$q_*(s,a) = \mathcal R^a_s + \gamma \sum_{s'} \mathcal P^a_{ss'} \max_{a'} q_*(s',a'). \tag{8}$$

After taking $a$ and reaching $s'$, the formula assumes we choose the next best action according to $q_*$. That is why $\max_{a'}$ sits inside the sum. **Why not take the max over the next state?** The agent chooses actions; it cannot choose the state that the environment generates randomly; so we still have to take the expectation under $\mathcal P^a_{ss'}$. $q_*$ describes the optimal outcome of the MDP, different from an estimate $Q$ that is being learned.

In a finite discounted MDP, there exists a stationary deterministic optimal policy. This does not say that a neural policy optimizer will find that policy. Nor does the statement carry over unchanged to partial observability or other objectives.

> **Code.** `bell.q_star` repeats the backup of Eq. (8). In this example $q_*(s,a) = q_\pi(s,a)$ because $u$ behaves the same for every policy, yet $v_*(s) = \max_a q_*(s,a) = 4.7 > v_\pi(s) = 3.85$: evaluation (average) vs optimality (max).

In [5]:
q_star = bell.q_star()
print("q_*(s, .) =", q_star[s].round(3), "-> v_*(s) =", q_star[s].max().round(3), "| v_pi(s) =", v[s].round(3))
print("greedy action w.r.t. q_* in s:", mdp.action_names[int(q_star[s].argmax())])

q_*(s, .) = [4.7 3. ] -> v_*(s) = 4.7 | v_pi(s) = 3.85
greedy action w.r.t. q_* in s: a


## 4. Example and pipeline

State $s$ has action $a$ that gives reward $2$, then goes to $u$ with probability $0.75$, to the terminal with $0.25$. If $v_\pi(u) = 4$, $\gamma = 0.9$, then $q_\pi(s,a) = 2 + 0.9(0.75\cdot 4) = 4.7$. Action $b$ ends immediately with reward $3$ so $q_\pi(s,b) = 3$. A policy that chooses the two actions evenly has $v_\pi(s) = 3.85$.

1. **Data:** If the model is known, use the exact expectation; if not, take sample transitions.
2. **Learning:** Estimate a value that satisfies the Bellman relation by backups or learning updates.
3. **Use and evaluation:** Greedy with respect to $q_*$ achieves optimal control in the setting above; greedy with respect to a $Q$ being learned may be wrong.

In [6]:
print(f"q_pi(s,a) = 2 + 0.9 * (0.75 * 4) = {q[s, 0]:.2f}")
print(f"q_pi(s,b) = {q[s, 1]:.2f}")
print(f"v_pi(s) = 0.5 * {q[s, 0]:.2f} + 0.5 * {q[s, 1]:.2f} = {v[s]:.2f}")

q_pi(s,a) = 2 + 0.9 * (0.75 * 4) = 4.70
q_pi(s,b) = 3.00
v_pi(s) = 0.5 * 4.70 + 0.5 * 3.00 = 3.85


## 5. Comparison, limits and applications

$q_\pi$ is not equal to $q_*$ unless the policy is good enough. A small Bellman error on the data that has been seen does not check every state–action. Value is useful for choosing routes, allocating actions and serving as a critic; it is not a success probability unless the reward/horizon is designed to have that meaning.

## 6. Exercises

1. If in the example $v_\pi(u) = 2$, what are $q_\pi(s,a)$ and $v_\pi(s)$?
2. Why does being greedy with respect to $q_\pi$ change the policy and therefore the value?
3. With only $v_\pi$ but no model: can we always choose the greedy action?

> **Code.** Exercise 1 rebuilt with $v_\pi(u) = 2$; exercise 2: acting greedily on $q_\pi$ gives a new policy whose value is different.

In [7]:
mdp2 = lecture_example_mdp(v_u=2.0)
bell2 = BellmanEquations(mdp2)
v2 = bell2.evaluate(uniform_policy(mdp2)); q2 = bell2.q_from_v(v2)
print(f"Exercise 1: q_pi(s,a) = {q2[s, 0]:.3f}, v_pi(s) = {v2[s]:.3f}")

greedy = TabularPolicy.greedy(q2).table
print("Exercise 2: greedy action in s =", mdp2.action_names[int(greedy[s].argmax())],
      f"-> new value v(s) = {mdp2.evaluate_policy_exact(greedy)[s]:.3f} (was {v2[s]:.3f})")

Exercise 1: q_pi(s,a) = 3.350, v_pi(s) = 3.175
Exercise 2: greedy action in s = a -> new value v(s) = 3.350 (was 3.175)


<details>
<summary><b>Answers (Appendix A)</b></summary>

(1) $q_\pi(s,a) = 2 + 0.9\cdot 0.75\cdot 2 = 3.35$; the uniform policy has $v_\pi(s) = (3.35 + 3)/2 = 3.175$. (2) Greedy changes the action probabilities, so the future trajectories and the value also change. (3) Not in general; a model/reward is needed to look ahead, or an appropriate action-value/actor.
</details>

## 7. Remember and read further

Bellman is a recursive expectation, not by default a TD loss. Use the expectation to evaluate a policy, use the max to describe optimality. Read Sutton–Barto §3.5 and then move on to DP.